# Aula 07 — NumPy: Arrays e Indexação Vetorial

**Semana 4 | 50 min | Referências: VanderPlas, *Python Data Science Handbook*, Cap. 2 · McKinney, *Python for Data Analysis*, Cap. 4**

## 🎯 Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- Explicar por que `np.ndarray` é dezenas de vezes mais rápido que listas em laços numéricos;
- Criar arrays com `np.array`, `arange`, `linspace`, `zeros`, `full` e `default_rng` (com semente fixa);
- Ler e escrever indexação/slicing 2-D (`fat[linha, mês]`, `fat[:, -12:]`) sem hesitar;
- Dizer quando um slicing devolve **view** e quando devolve **cópia** — e proteger análises com `.copy()`;
- Montar a matriz de faturamento mensal de 6 filiais (36 meses) que a Aula 08 vai deflacionar.

## 1. Por que NumPy? Lista vs ndarray

**Intuição.** Sua planilha de faturamento tem 6 filiais × 36 meses = 216 números.
Com listas Python, cada número é um objeto completo em memória, e qualquer operação
exige um laço escrito por você. Com `ndarray`, os 216 números vivem num bloco contíguo
com um único tipo (`dtype`) — e operações como `fat * 1.05` aplicam-se a **todos** os
elementos de uma vez, em código compilado.

Quanto mais rápido? Vamos medir a diferença com `%timeit`: somar 1 milhão de números
das duas maneiras.

In [1]:
# Setup — imports e padrões visuais usados na aula
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

def brl(x):
    """Formata um número como moeda brasileira: R$ 1.234,56."""
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

np.random.seed(42)  # reprodutibilidade quando houver aleatoriedade

In [2]:
# Mesma soma de 1 milhão de números: versão lista (laço implícito do sum) vs NumPy
numeros = list(range(10**6))
arr = np.arange(10**6)

print("Lista (sum do Python):")
%timeit sum(numeros)

print("NumPy (np.sum):")
%timeit arr.sum()

Lista (sum do Python):


2.66 ms ± 4.45 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
NumPy (np.sum):


92.6 μs ± 2.25 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


**Leitura do resultado.** `np.sum` costuma ser **30–100× mais rápido** que `sum`
sobre a lista. O ganho vem de onde o laço mora: no Python ele roda interpretado, no
NumPy roda compilado em C sobre memória contígua. Em análises com milhões de
observações (microdados da POF, séries diárias do BCB), essa diferença é a diferença
entre segundos e minutos — e entre usar e não usar.

## 2. Criando a matriz de faturamento

**Intuição.** Vamos montar a matriz que acompanha as duas aulas: **6 filiais de Goiás**
(Goiânia Campinas, Goiânia Bueno, Anápolis, Rio Verde, Jataí, Catalão) × **36 meses de
faturamento nominal** (ago/2023 → jul/2026). Filiais maiores partem de uma base maior e
crescem mais devagar (maturidade); filiais novas partem pequenas e crescem rápido.
Os valores são sintéticos, mas calibrados com números plausíveis para o varejo agro
goiano — e a semente fixa (`42`) garante que todos gerem exatamente a mesma matriz.

In [3]:
# Nomes das filiais e calendário: 36 meses de ago/2023 a jul/2026
import itertools

filiais = ["Goiânia Campinas", "Goiânia Bueno", "Anápolis", "Rio Verde", "Jataí", "Catalão"]

# rótulos "mmm/aaaa" para consultar o calendário sem laço mental
mes_num = np.arange(36)
ano = 2023 + (7 + mes_num) // 12          # ago/2023 é o mês 0
mes_do_ano = ((7 + mes_num) % 12) + 1     # 1 = janeiro ... 12 = dezembro
meses = np.array([f"{m:02d}/{a}" for m, a in zip(mes_do_ano, ano)])
meses[:3], meses[-3:]

(array(['08/2023', '09/2023', '10/2023'], dtype='<U7'),
 array(['05/2026', '06/2026', '07/2026'], dtype='<U7'))

In [4]:
# Matriz de faturamento sintético 6 × 36: base × tendência × sazonalidade × ruído
rng = np.random.default_rng(42)   # gerador moderno do NumPy, com semente fixa

base = np.array([1_350_000, 920_000, 680_000, 430_000, 190_000, 88_000], dtype=float)
cresc = np.array([0.004, 0.003, 0.005, 0.006, 0.008, 0.015])   # crescimento mensal por filial
m = np.arange(36)

tendencia = (1 + cresc[:, None]) ** m                 # (6, 1) elevado a (36,) -> (6, 36)
sazonalidade = 1 + 0.06 * ((m % 12) >= 10)            # +6% nos meses de safra (nov–dez)
ruido = rng.normal(0, 0.05, size=(6, 36))             # choques de ±5% típicos do varejo

fat = np.round(base[:, None] * tendencia * sazonalidade * (1 + ruido), 2)
fat.shape

(6, 36)

In [5]:
# Conferindo a matriz: primeira e última filial nos primeiros meses (valores em R$)
fat[:2, :4]

array([[1370568.4 , 1284920.28, 1411883.11, 1430517.91],
       [ 914758.42,  883996.86,  887374.25,  958502.29]])

**Leitura do resultado.** `fat.shape == (6, 36)`: linhas = filiais, colunas = meses.
Goiânia Campinas parte de ~R$ 1,35 mi/mês; a Catalão, filial nova, de ~R$ 88 mil.
Note que os cálculos usaram **arrays 2-D com `(6, 1)` e `(36,)`** combinados sem laço —
um prévia do broadcasting que a Aula 8 formaliza.

## 3. `dtype`: o tipo único da matriz

**Intuição.** NumPy guarda a matriz inteira num único formato numérico — o `dtype`.
Nossa matriz é `float64` (padrão para valores monetários). Converter tipos
(*casting*) é às vezes necessário, mas o casting para inteiro **trunca**: R$ 98.999,70
vira 98999. Em dinheiro, quase sempre é o que você **não** quer.

In [6]:
# dtype: o formato único de todos os elementos
print("dtype da matriz:", fat.dtype)

# casting explícito: cuidado com truncamento em valores monetários
print("fat.astype(int) trunca, por exemplo:", fat[0, 0], "->", int(fat[0, 0]))

# o jeito certo de ver um número 'feio': arredondar para exibir, não truncar
np.round(fat[0, 0], 2)

dtype da matriz: float64
fat.astype(int) trunca, por exemplo: 1370568.4 -> 1370568


np.float64(1370568.4)

## 4. Indexação e slicing vetorial

**Intuição.** A pergunta do analista quase nunca é "qual é o elemento (0, 0)?", e sim:
*"como está o último ano?", "e a Catalão?", "e o mês de dezembro?". Slicing responde
a todas sem laço. Sintaxe: `fat[linhas, colunas]`, com `:` = "todos", `a:b` =
"de a até b−1" e índices negativos contando do fim.

In [7]:
# Um elemento, uma linha, uma coluna: as três leituras básicas
print("Goiânia Campinas, mês ago/2023 :", brl(fat[0, 0]))
print("Linha inteira = a Catalão nos 36 meses; shape:", fat[5].shape)
print("Coluna inteira = todos os 6 filiais em jan/2024 (mês 5):")
fat[:, 5]

Goiânia Campinas, mês ago/2023 : R$ 1.370.568,40
Linha inteira = a Catalão nos 36 meses; shape: (36,)
Coluna inteira = todos os 6 filiais em jan/2024 (mês 5):


array([1287547.69,  959245.18,  702697.31,  411021.52,  199275.18,
        102080.55])

In [8]:
# Janelas de tempo: primeiro ano, último ano, trimestre final
primeiro_ano = fat[:, :12]
ultimo_ano = fat[:, -12:]
tri_final = fat[:, -3:]

print("média mensal da rede no 1º ano :", brl(primeiro_ano.mean()))
print("média mensal da rede no último ano:", brl(ultimo_ano.mean()))
print("shape do trimestre final:", tri_final.shape)

média mensal da rede no 1º ano : R$ 630.695,26
média mensal da rede no último ano: R$ 706.827,39
shape do trimestre final: (6, 3)


In [9]:
# Passo e reversão: filiais alternadas a cada 6 meses; filiais em ordem inversa
fat[::2, ::6]

array([[1370568.4 , 1391564.14, 1420920.56, 1514289.33, 1453921.01,
        1684709.76],
       [ 648738.62,  722572.91,  678774.63,  768619.74,  715778.26,
         775684.05],
       [ 182277.06,  197723.53,  210426.14,  242641.21,  228239.93,
         225888.05]])

**Leitura do resultado.** `fat[::2, ::6]` pega filiais 0, 2, 4 (Goiânia Campinas,
Anápolis, Jataí) nos meses 0, 6, 12, 18, 24, 30 — subamostragem para uma olhada
rápida. `fat[::-1]` inverte a ordem das filiais. A regra geral: cada dimensão aceita
`início:fim:passo`, e omitir significa "tudo".

## 5. Views vs cópias: o ponto crítico da aula

**Intuição.** Quando você recorta com slicing simples, o NumPy **não copia nada**:
devolve uma *view* — outra janela para a mesma memória. Ótimo para desempenho,
perigoso para análises: **alterar a view altera a matriz original**, sem qualquer
aviso. Veja a contaminação acontecer (e como evitá-la).

In [10]:
# Demonstração do problema: slice -> view -> mutação contamina a original
janela = fat[:, -6:]        # view dos últimos 6 meses (sem .copy())

antes = fat[0, 30].copy()   # guarda só o número para comparar (cópia de um escalar)
janela[0, 0] = -1.0         # ⚠ escrevemos "na janela"...

print("valor antes :", brl(antes))
print("fat[0, 30]  :", brl(fat[0, 30]), " <- a original MUDOU!")
print("fat.base da janela é a própria fat?", janela.base is not None)

valor antes : R$ 1.684.709,76
fat[0, 30]  : R$ -1,00  <- a original MUDOU!
fat.base da janela é a própria fat? True


In [11]:
# Reparo: restauramos o valor e refazemos o recorte COM .copy()
janela[0, 0] = antes            # conserta a matriz original (a view escreve nela)
fat[0, 30] = antes              # garantia dupla

janela_segura = fat[:, -6:].copy()   # agora sim: memória nova, independente
janela_segura[0, 0] = 999_999.99

print("janela_segura[0,0]:", brl(janela_segura[0, 0]))
print("fat[0, 30]        :", brl(fat[0, 30]), " <- intocado: a cópia protegeu a análise")

janela_segura[0,0]: R$ 999.999,99
fat[0, 30]        : R$ 1.684.709,76  <- intocado: a cópia protegeu a análise


**Regra do curso.** *Recortou para transformar de forma independente? `.copy()`.*
Ler um recorte pode ficar como view; **escrever** em um recorte exige cópia (ou
intenções explícitas de editar a original). Fancy indexing (`fat[[0, 2, 4]]`) e
máscaras booleanas devolvem cópias — o perigo está no slice simples.

## 6. 📝 Exercícios

Os exercícios usam a matriz `fat` (6 × 36) e a lista `filiais` criadas acima.
Tente resolver **antes** de abrir a célula `# SOLUÇÃO N` logo abaixo.

### Exercício 1 — Janela de safra

A diretoria pediu a média de faturamento da **rede inteira** (todas as filiais)
nos **últimos 12 meses**. Usando slicing, guarde os últimos 12 meses em `ultimo_ano`
e exiba a média com `brl()`.

In [12]:
# EXERCÍCIO 1 — seu código aqui



In [13]:
# SOLUÇÃO 1
ultimo_ano = fat[:, -12:]        # todas as linhas (filiais), 12 meses finais
media_ultimo_ano = ultimo_ano.mean()
brl(media_ultimo_ano)

'R$ 706.827,39'

### Exercício 2 — Uma filial em mãos

Extraia a **linha inteira** da filial Rio Verde (índice 3) e calcule o faturamento
total dos 36 meses, exibindo com `brl()`.

In [14]:
# EXERCÍCIO 2 — seu código aqui



In [15]:
# SOLUÇÃO 2
rio_verde = fat[3]               # linha inteira (mesmo que fat[3, :])
total_rio_verde = rio_verde.sum()
brl(total_rio_verde)

'R$ 17.382.376,52'

### Exercício 3 — Meta mensal e cumprimento

Crie uma matriz de metas `meta` com `np.full((6, 36), 100_000.0)` (R$ 100 mil/mês por
filial) e calcule **em que fração dos meses-filial** a meta foi batida
(`fat > meta`, depois `.mean()` sobre o booleano).

In [16]:
# EXERCÍCIO 3 — seu código aqui



In [17]:
# SOLUÇÃO 3
meta = np.full((6, 36), 100_000.0)
atingiu = fat > meta             # booleano 6 × 36: True = meta batida no mês
pct_atingiu = atingiu.mean()     # fração de True (True=1, False=0)
print(f"{pct_atingiu:.1%} dos meses-filial bateram a meta de R$ 100 mil")

96.3% dos meses-filial bateram a meta de R$ 100 mil


### Exercício 4 — Cópia defensiva

Um mês da Catalão (índice de linha 5) foi lançado **em duplicidade** e precisa ser
zerado numa análise, **sem alterar a matriz original** `fat`. Recorte a linha com
cópia, zere o mês de índice 10 e confirme que `fat[5, 10]` permanece com o valor
original.

In [18]:
# EXERCÍCIO 4 — seu código aqui



In [19]:
# SOLUÇÃO 4
catalao = fat[5].copy()          # SEM .copy() a correção sujaria a matriz original
valor_original = fat[5, 10]
catalao[10] = 0.0                # zera o lançamento duplicado SÓ na cópia

print("cópia corrigida :", brl(catalao[10]))
print("fat[5, 10]      :", brl(fat[5, 10]), " <- intocado; nada mudou na matriz original")

cópia corrigida : R$ 0,00
fat[5, 10]      : R$ 117.585,04  <- intocado; nada mudou na matriz original


## 📌 Resumo & para casa

- `ndarray` = bloco contíguo com `dtype` único: a mesma operação custa 30–100× menos
  que o laço em listas (`np.sum` vs `sum` em 10⁶ números).
- Criação essencial: `np.array`, `arange`, `linspace`, `zeros`, `full`,
  `rng = np.random.default_rng(42)` (sempre com semente).
- Indexação 2-D: `fat[linha, mês]`; janelas com `fat[:, -12:]`; passo e reversão com
  `::2`, `::-1`.
- **Views vs cópias** é o ponto de segurança da aula: slicing devolve view — escrever
  nela altera a original; `.copy()` compra isolamento.
- Casting para inteiro trunca; para exibir, `np.round`; para dinheiro, `float64`.
- **Para casa**: refazer a demonstração da Seção 5 de memória e prever, linha a linha,
  o que aparece no print antes de rodar.
- **Próxima aula (A08)**: a matriz `fat` desta aula é deflacionada pelo IPCA real
  (BCB) com ufuncs e broadcasting — uma linha para 216 divisões.

Referências (KB):
- `kb/02_vanderplas_python_data_science_handbook/05_chapter-2-introduction-to-numpy.md` — Cap. 2.
- `kb/01_mckinney_python_for_data_analysis/06_chapter-4-numpy-basics-arrays-and-vectorized-computation.md` — Cap. 4.